# 🧠 Prototype 3 — Stage 2: Cascade Model Training

**Architecture:** Cascaded PanT-HybridNet = Swin-UNETR + CBAM3D Attention

**Why this works now:** Our Swin-UNETR is no longer seeing the full body (where tumour = 0.5% of voxels).
It only sees the tiny cropped Pancreas region (where tumour = 10–15% of voxels).
Class imbalance is mathematically destroyed. Loss will converge properly.

**Training physics:**
- Optimizer: `AdamW` (decoupled weight decay — essential for Transformers)
- LR Schedule: Linear Warmup (10 epochs) → Cosine Annealing decay
- Loss: `DiceFocalLoss` (aggressively penalizes missing the tumour)
- Gradient Clipping: `max_norm=1.0` (prevents explosion)

In [ ]:
# Install dependencies (run this FIRST — only needed once per Colab session)
!pip install -q monai[all] nibabel tqdm
print("✅ All packages installed!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5

In [ ]:
# ============================================================
# CELL 1: Setup
# ============================================================
import os, glob, json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from monai.networks.nets import SwinUNETR
from monai.losses import DiceFocalLoss
import monai.transforms as mt
from monai.inferers import sliding_window_inference
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Correct Drive paths (verified from your output)
CASCADE_DIR = '/content/drive/MyDrive/PanTS_Cascade'
SAVE_DIR    = '/content/drive/MyDrive/PanT_Cascade_Model'
os.makedirs(SAVE_DIR, exist_ok=True)

patient_folders = sorted(glob.glob(f'{CASCADE_DIR}/PanTS_*'))
print(f'\nCascade patients found: {len(patient_folders)}')
for p in patient_folders:
    info = json.load(open(f'{p}/bbox.json'))
    print(f"  {info['pid']} | cropped: {info['cropped_shape']} | tumour: {info['has_tumor']}")

if len(patient_folders) == 0:
    raise RuntimeError('No cascade patients found! Run 01_Data_Preparation_3.ipynb first.')

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Mounted at /content/drive
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB

Cascade patients found: 4
  PanTS_00000442 | cropped: [229, 126, 158] | tumour: False
  PanTS_00000717 | cropped: [208, 85, 119] | tumour: False
  PanTS_00000806 | cropped: [137, 112, 61] | tumour: True
  PanTS_00000871 | cropped: [162, 125, 135] | tumour: True


In [ ]:
# ============================================================
# CELL 2: Dataset & DataLoader
# ============================================================
train_transforms = mt.Compose([
    mt.EnsureChannelFirstd(keys=['image','label'], channel_dim='no_channel'),
    mt.ScaleIntensityRanged(keys=['image'], a_min=-100, a_max=240,
                             b_min=0.0, b_max=1.0, clip=True),
    mt.Spacingd(keys=['image','label'], pixdim=(1.5,1.5,1.5),
                mode=('bilinear','nearest')),
    mt.SpatialPadd(keys=['image','label'], spatial_size=(64,64,64)),
    # ratios=[bg, pancreas, tumor] — 50% of patches FORCED to overlap tumor!
    mt.RandCropByLabelClassesd(
        keys=['image','label'], label_key='label',
        spatial_size=(64,64,64),
        ratios=[0.1, 0.4, 0.5],   # 10% bg, 40% pancreas, 50% tumor
        num_classes=3,
        num_samples=6,
        warn=False
    ),
    mt.RandFlipd(keys=['image','label'], prob=0.5, spatial_axis=0),
    mt.RandFlipd(keys=['image','label'], prob=0.5, spatial_axis=1),
    mt.RandFlipd(keys=['image','label'], prob=0.5, spatial_axis=2),
    mt.EnsureTyped(keys=['image','label'])
])

class CascadeDataset(Dataset):
    def __init__(self, folders, transform=None):
        self.folders = folders
        self.transform = transform

    def __len__(self):
        return len(self.folders)

    def __getitem__(self, idx):
        folder = self.folders[idx]
        ct  = nib.load(f'{folder}/ct_cropped.nii.gz').get_fdata().astype(np.float32)
        lbl = nib.load(f'{folder}/label_cropped.nii.gz').get_fdata().astype(np.float32)
        if self.transform:
            return self.transform({'image': ct, 'label': lbl})
        return {'image': ct, 'label': lbl}

def collate_patches(batch):
    imgs   = torch.stack([p['image'] for patches in batch for p in patches])
    labels = torch.stack([p['label'] for patches in batch for p in patches])
    return {'image': imgs, 'label': labels}

train_ds     = CascadeDataset(patient_folders, transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          collate_fn=collate_patches, num_workers=2)
print(f'DataLoader ready: {len(train_ds)} patients')
print('Patch sampling: 10% background | 40% pancreas | 50% TUMOR-FORCED')


DataLoader ready: 4 patients
Patch sampling: 10% background | 40% pancreas | 50% TUMOR-FORCED


In [ ]:
class CBAM3D(nn.Module):
    """Channel + Spatial Attention to focus on tumour boundaries."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(1, channels // reduction)  # ← FIX: never allow 0 channels
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.max_pool = nn.AdaptiveMaxPool3d(1)
        self.fc = nn.Sequential(
            nn.Conv3d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv3d(mid, channels, 1, bias=False)
        )
        self.spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        ca = self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
        x  = x * ca
        sa = self.sigmoid(self.spatial(torch.cat([
            x.mean(dim=1, keepdim=True),
            x.max(dim=1,  keepdim=True)[0]
        ], dim=1)))
        return x * sa

class CascadedPanTHybridNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.swin = SwinUNETR(
            in_channels=1, out_channels=num_classes,
            feature_size=24, use_checkpoint=True
        )
        self.cbam = CBAM3D(channels=num_classes)

    def forward(self, x):
        out = self.swin(x)
        if isinstance(out, (list, tuple)): out = out[0]
        return self.cbam(out)

model = CascadedPanTHybridNet(num_classes=3).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: CascadedPanTHybridNet (Swin-UNETR + CBAM3D)')
print(f'Trainable params: {total_params:,}')


Model: CascadedPanTHybridNet (Swin-UNETR + CBAM3D)
Trainable params: 15,703,721


In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 150

# Class weights force model to stop predicting background
# Background=0.1 (penalize less), Pancreas=2.0, Tumour=8.0 (penalize heavily)
class_weights = torch.tensor([0.1, 2.0, 8.0]).to(device)

from monai.losses import DiceLoss
dice_loss_fn = DiceLoss(include_background=False, softmax=True, to_onehot_y=True)
ce_loss_fn   = nn.CrossEntropyLoss(weight=class_weights)

def combined_loss(preds, labels):
    dl = dice_loss_fn(preds, labels)
    cl = ce_loss_fn(preds, labels.squeeze(1))
    return dl + 0.5 * cl

# Fresh optimizer — higher LR for 4-patient memorization
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda')

print('Loss:      DiceLoss + Weighted CrossEntropy [BG=0.1, Panc=2.0, Tumor=8.0]')
print('Optimizer: AdamW  lr=3e-4')
print('Scheduler: CosineAnnealing  (no warmup — 4 patients need aggressive learning)')


Loss:      DiceLoss + Weighted CrossEntropy [BG=0.1, Panc=2.0, Tumor=8.0]
Optimizer: AdamW  lr=3e-4
Scheduler: CosineAnnealing  (no warmup — 4 patients need aggressive learning)


In [ ]:
from monai.transforms import AsDiscrete
post_pred = AsDiscrete(argmax=True)

def dice_np(pred_np, gt_np, cls):
    p = (pred_np == cls).astype(np.float32)
    g = (gt_np   == cls).astype(np.float32)
    inter = np.sum(p * g)
    return (2.0 * inter) / (np.sum(p) + np.sum(g) + 1e-6)

# No Spacingd here — compare pred and label in the same native space
val_transforms = mt.Compose([
    mt.EnsureChannelFirstd(keys=['image','label'], channel_dim='no_channel'),
    mt.ScaleIntensityRanged(keys=['image'], a_min=-100, a_max=240,
                             b_min=0.0, b_max=1.0, clip=True),
    mt.EnsureTyped(keys=['image','label'])
])

best_loss = float('inf')

print('=' * 75)
print('  STAGE 2: CASCADE FINE-STAGE TRAINING (Swin-UNETR + CBAM3D)')
print('=' * 75)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss, steps = 0.0, 0

    for batch in train_loader:
        imgs   = batch['image'].to(device)
        labels = batch['label'].long().to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss  = loss_fn(preds, labels)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
        steps += 1

    scheduler.step()
    avg_loss = epoch_loss / max(steps, 1)
    cur_lr   = scheduler.get_last_lr()[0]

    saved = ''
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({'epoch': epoch+1, 'model': model.state_dict(),
                    'loss': best_loss}, f'{SAVE_DIR}/cascade_best.pth')
        saved = '  💾 SAVED BEST'

    # Validation DSC every 10 epochs
    if (epoch+1) % 10 == 0 or epoch == 0:
        model.eval()
        panc_dscs, tumor_dscs = [], []
        with torch.no_grad():
            for pf in patient_folders:
                ct  = nib.load(f'{pf}/ct_cropped.nii.gz').get_fdata().astype(np.float32)
                lbl = nib.load(f'{pf}/label_cropped.nii.gz').get_fdata().astype(np.uint8)

                # Transform both image and label together so shapes always match
                prepped = val_transforms({'image': ct, 'label': lbl.astype(np.float32)})
                inp     = prepped['image'].unsqueeze(0).to(device)  # (1,1,H,W,D)
                lbl_np  = prepped['label'].numpy().squeeze(0).astype(np.uint8)  # (H,W,D)

                with torch.amp.autocast('cuda'):
                    raw = sliding_window_inference(inp, (64,64,64), 2, model)
                # post_pred output: (1,H,W,D) — squeeze channel dim
                pred_np = post_pred(raw.squeeze(0)).cpu().numpy().squeeze(0)  # (H,W,D)

                panc_dscs.append(dice_np(pred_np, lbl_np, 1))
                if np.sum(lbl_np == 2) > 0:
                    tumor_dscs.append(dice_np(pred_np, lbl_np, 2))

        p_dsc = np.mean(panc_dscs) * 100
        t_dsc = np.mean(tumor_dscs) * 100 if tumor_dscs else 0.0
        print(f'Epoch {epoch+1:>3}/{EPOCHS}  |  Loss: {avg_loss:.4f}  |  LR: {cur_lr:.6f}  |  Panc: {p_dsc:5.1f}%  |  Tumor: {t_dsc:5.1f}%{saved}')
    else:
        print(f'Epoch {epoch+1:>3}/{EPOCHS}  |  Loss: {avg_loss:.4f}  |  LR: {cur_lr:.6f}{saved}')
from monai.transforms import AsDiscrete
post_pred = AsDiscrete(argmax=True)

def dice_np(pred_np, gt_np, cls):
    p = (pred_np == cls).astype(np.float32)
    g = (gt_np   == cls).astype(np.float32)
    inter = np.sum(p * g)
    return (2.0 * inter) / (np.sum(p) + np.sum(g) + 1e-6)

val_transforms = mt.Compose([
    mt.EnsureChannelFirstd(keys=['image','label'], channel_dim='no_channel'),
    mt.ScaleIntensityRanged(keys=['image'], a_min=-100, a_max=240,
                             b_min=0.0, b_max=1.0, clip=True),
    mt.EnsureTyped(keys=['image','label'])
])

best_loss = float('inf')

print('=' * 75)
print('  STAGE 2: CASCADE TRAINING — WEIGHTED LOSS (Background Mode Fixed)')
print('=' * 75)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss, steps = 0.0, 0

    for batch in train_loader:
        imgs   = batch['image'].to(device)
        labels = batch['label'].long().to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss  = combined_loss(preds, labels)   # ← uses weighted loss
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
        steps += 1

    scheduler.step()
    avg_loss = epoch_loss / max(steps, 1)
    cur_lr   = scheduler.get_last_lr()[0]

    saved = ''
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({'epoch': epoch+1, 'model': model.state_dict(),
                    'loss': best_loss}, f'{SAVE_DIR}/cascade_best.pth')
        saved = '  💾 SAVED BEST'

    if (epoch+1) % 10 == 0 or epoch == 0:
        model.eval()
        panc_dscs, tumor_dscs = [], []
        with torch.no_grad():
            for pf in patient_folders:
                ct  = nib.load(f'{pf}/ct_cropped.nii.gz').get_fdata().astype(np.float32)
                lbl = nib.load(f'{pf}/label_cropped.nii.gz').get_fdata().astype(np.uint8)
                prepped = val_transforms({'image': ct, 'label': lbl.astype(np.float32)})
                inp     = prepped['image'].unsqueeze(0).to(device)
                lbl_np  = prepped['label'].numpy().squeeze(0).astype(np.uint8)
                with torch.amp.autocast('cuda'):
                    raw = sliding_window_inference(inp, (64,64,64), 2, model)
                pred_np = post_pred(raw.squeeze(0)).cpu().numpy().squeeze(0)
                panc_dscs.append(dice_np(pred_np, lbl_np, 1))
                if np.sum(lbl_np == 2) > 0:
                    tumor_dscs.append(dice_np(pred_np, lbl_np, 2))

        p_dsc = np.mean(panc_dscs) * 100
        t_dsc = np.mean(tumor_dscs) * 100 if tumor_dscs else 0.0
        print(f'Epoch {epoch+1:>3}/{EPOCHS}  |  Loss: {avg_loss:.4f}  |  LR: {cur_lr:.6f}  |  Panc: {p_dsc:5.1f}%  |  Tumor: {t_dsc:5.1f}%{saved}')
    else:
        print(f'Epoch {epoch+1:>3}/{EPOCHS}  |  Loss: {avg_loss:.4f}  |  LR: {cur_lr:.6f}{saved}')

print()
print('=' * 75)
print(f'  DONE!  Best Loss: {best_loss:.4f}')
print(f'  Checkpoint: {SAVE_DIR}/cascade_best.pth')
print('=' * 75)

print()
print('=' * 75)
print(f'  TRAINING COMPLETE!  Best Loss: {best_loss:.4f}')
print(f'  Checkpoint: {SAVE_DIR}/cascade_best.pth')
print('=' * 75)


  STAGE 2: CASCADE FINE-STAGE TRAINING (Swin-UNETR + CBAM3D)
Epoch   1/150  |  Loss: 1.1132  |  LR: 0.000300  |  Panc:  13.2%  |  Tumor:   0.6%  💾 SAVED BEST
Epoch   2/150  |  Loss: 1.1121  |  LR: 0.000300  💾 SAVED BEST
Epoch   3/150  |  Loss: 1.1026  |  LR: 0.000300  💾 SAVED BEST
Epoch   4/150  |  Loss: 1.0905  |  LR: 0.000299  💾 SAVED BEST
Epoch   5/150  |  Loss: 1.0915  |  LR: 0.000299
Epoch   6/150  |  Loss: 1.0812  |  LR: 0.000299  💾 SAVED BEST
Epoch   7/150  |  Loss: 1.0806  |  LR: 0.000298  💾 SAVED BEST
Epoch   8/150  |  Loss: 1.0695  |  LR: 0.000298  💾 SAVED BEST
Epoch   9/150  |  Loss: 1.0605  |  LR: 0.000297  💾 SAVED BEST
Epoch  10/150  |  Loss: 1.0483  |  LR: 0.000297  |  Panc:  16.2%  |  Tumor:   0.2%  💾 SAVED BEST
Epoch  11/150  |  Loss: 1.0578  |  LR: 0.000296
Epoch  12/150  |  Loss: 1.0454  |  LR: 0.000295  💾 SAVED BEST
Epoch  13/150  |  Loss: 1.0443  |  LR: 0.000294  💾 SAVED BEST
Epoch  14/150  |  Loss: 1.0203  |  LR: 0.000294  💾 SAVED BEST
Epoch  15/150  |  Loss: 1.027